In [100]:
import pandas as pd
import numpy as np
from collections import defaultdict
import json

In [24]:
clues = pd.read_csv("clues_clean.csv").drop("Unnamed: 0", axis=1)

In [27]:
clues.sample(10)

,index,answer,clue_1,clue_2,clue_3,clue_4,clue_5,clue_6,clue_7,clue_8,clue_9,clue_10
2493,2493,ASPCA,Adoption org.,Humane org.,Humane org. since 1866,Org. that facilitates adoptions,Pet adoption org.,Pet protection org.,Pet protection org.,"Pet protector, for short",Pet welfare grp.,We Are Their Voice org.
15289,15289,LAM,Escape,Escapee's flight,Flight,Head for the hills,Hurried flight,Make a run for it,On the ___ (fleeing),Take a powder,Taking it on the ___ (fleeing),Thrash
20122,20122,OPENTO,Entertaining,Interested in trying,Not dismissive of,Not entirely against,Not ruling out,Not totally against,Receptive,Willing to consider,Willing to try,___ the public
19309,19309,OATS,Cereal grains,Horse feed,Mare fare,Mare's meal,Mares eat them,Overnight ___,Part of a stable diet,Quaker ___,Stable staples,Texas is the largest U.S. producer of these
31413,31413,WAHOOS,Cowboy cries,Cries at a Wild West show,Elated cheers,Exuberant cries,Exuberant shouts,Exuberant-sounding mackerels?,"Large, swift mackerel",Shouts heard at a rodeo,"University of Virginia players, familiarly",Warm-water mackerels
28914,28914,TENANTS,Apartment dwellers,Dwellers in complexes,Flat people?,Lease signatories,Many urban dwellers,Monthly rent payers,Rent strike participants,Security deposit payers,"The Ricardos, to the Mertzes",They're in your apartment building
15881,15881,LETSOUT,Divulges,"Ends, as a class","Ends, as class","Frees for a bit, catlover's style",Frees from prison,Paroles,Paroles,Releases,Releases,Spills the beans
21721,21721,PLAYPEN,Baby's enclosure,Corral for a kiddie,Infant's recreation area,Kiddy coop,Place to put some toys,Toddler's enclosure,Toddlers' enclosure,Tot spot,Tot's enclosure,Tot's turf
25286,25286,SEEM,Appear,Appear,Appear,Appear to be,Come across as,Come off as,Come off as,Give the impression of being,Have the appearance of,Look (like)
14312,14312,ITAS,But I must also feel ___ a man: Macduff,Take ___ a blessing,Take ___ a blessing,Take ___ a compliment,Take ___ a compliment,Take ___ a compliment,Take ___ a compliment,Take ___ a compliment!,Take ____ read,Take _____ a compliment


In [28]:
clues["length"]=[len(x) for x in clues.answer]

In [29]:
clues_by_length = {l:t for l,t in clues.groupby("length")}

In [32]:
clues_by_length[3]

,index,answer,clue_1,clue_2,clue_3,clue_4,clue_5,clue_6,clue_7,clue_8,clue_9,clue_10,length
0,0,AAA,Battery buy,Battery type,Battery type,Car club,Certain batteries,Driver's aid: Abbr.,Little battery,Roadside assistance org.,Rte. suggester,Small battery size,3
4,4,AAH,Contented sigh,Exclamation of amazement,Now I see!,Ooh and ___,Ooh's partner,Open-wide word,Something you might say at a doctor's office,Sound of delight,That feels g-o-o-o-d!,Word said to a doctor,3
13,13,AAR,Alpine stream,Berne's river,Rhine feeder,Rhine tributary of Switzerland,Swiss river,Swiss river,Swiss river,Swiss river into the Rhine,Swiss river to the Rhine,Switzerland's ___ Gorge,3
19,19,AAS,Certain batteries,"Remote batteries, often",Small batteries,Small batteries,Small batteries,Small batteries,Small batteries,Some batteries,Some flashlight batteries,___ in apple,3
20,20,AAU,Amateur sports org.,Athletes' grp.,Org. that bestows the Sullivan Award,Org. that gives the James E. Sullivan Award,"Org. with the Sullivan Award for character, le...","Org. with the Sullivan Award for character, le...",Org. with the annual Junior Olympic Games,"Sports For All, Forever org.",Sports org.,Sports org. for nonprofessionals,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32443,32443,ZIT,Bad spot,Bit of acne,Clearasil target,Oxy 5 target,Pimple,"Problem before a big date, informally","Skin blemish, slangily",Teen's facial blemish,Temple imperfection,Tough spot for a teen?,3
32446,32446,ZOA,Animal group suffix,"Animals, in taxonomy","Combining form meaning ""animals""",Ending with proto-,Organisms: Suffix,"Suffix meaning ""animals""","Suffix meaning ""animals""",Suffix with proto-,Suffix with proto-,Taxonomy suffix,3
32448,32448,ZOE,Actress Caldwell,Avatar actress Saldana,Four-time Tony winner Caldwell,High Fidelity actress Kravitz,"Orange ""Sesame Street"" Muppet",Playwright Akins,"Saldana of ""Avengers: Infinity War""","Saldana of ""Guardians of the Galaxy""",Tony winner Caldwell,Tony winner Caldwell,3
32456,32456,ZOO,Business with habitats,Chaotic place,Chaotic scene,Chaotic scene,Chaotic situation,"Dr. Seuss' ""If I Ran the ___""",Madhouse,Menagerie,Where the wild things are?,Workplace for some veterinarians,3


In [49]:
decompose_word("ZYDECO")

[['Z', 'Y', 'D', 'E', 'C', 'O'],
 ['Z', 'Y', 'D', 'E', 'CO'],
 ['Z', 'Y', 'D', 'CE', 'O'],
 ['Z', 'Y', 'DE', 'C', 'O'],
 ['Z', 'Y', 'DE', 'CO'],
 ['Z', 'DY', 'E', 'C', 'O'],
 ['Z', 'DY', 'E', 'CO'],
 ['Z', 'DY', 'CE', 'O'],
 ['YZ', 'D', 'E', 'C', 'O'],
 ['YZ', 'D', 'E', 'CO'],
 ['YZ', 'D', 'CE', 'O'],
 ['YZ', 'DE', 'C', 'O'],
 ['YZ', 'DE', 'CO']]

In [79]:
alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

In [89]:
character_set = list(alphabet)+[alphabet[i]+alphabet[j] for i in range(26) for j in range(i,26)]

In [90]:
len(character_set)

377

In [91]:
character_set[-1]

'ZZ'

In [92]:
char_table = {ch:n for n,ch in enumerate(character_set)}
idx_table = {n:ch for n,ch in enumerate(character_set)}

In [93]:
def decompose_word(w):
    if len(w) == 0:
        return []
    elif len(w) == 1:
        return [[w]]
    elif len(w) == 2:
        return [[w[0],w[1]],["".join(sorted(w))]]
    else:
        out = []
        first = w[0]
        first_suffix = decompose_word(w[1:])
        for i in first_suffix:
            out.append([first]+i)
        pair = "".join(sorted(w[:2]))
        second_suffix = decompose_word(w[2:])
        for i in second_suffix:
            out.append([pair]+i)
        return out

In [94]:
def decomp_to_idx(l):
    return tuple(char_table[i] for i in l)

In [95]:
[decomp_to_idx(x) for x in decompose_word("ZYDECO")]

[(25, 24, 3, 4, 2, 14),
 (25, 24, 3, 4, 89),
 (25, 24, 3, 79, 14),
 (25, 24, 102, 2, 14),
 (25, 24, 102, 89),
 (25, 122, 4, 2, 14),
 (25, 122, 4, 89),
 (25, 122, 79, 14),
 (375, 3, 4, 2, 14),
 (375, 3, 4, 89),
 (375, 3, 79, 14),
 (375, 102, 2, 14),
 (375, 102, 89)]

In [104]:
tags_to_word_index =[]
tags_by_length = defaultdict(list)
for i, w in zip(clues["index"], clues["answer"]):
    d = decompose_word(w)
    for t in d:
        tags = decomp_to_idx(t)
        tags_to_word_index.append((i,tags))
        tags_by_length[len(tags)].append(tags)
        

In [105]:
len(tags_by_length[5])

116792

In [106]:
len(tags_by_length)

14

In [107]:
with open("tags_by_length.json", "w") as f:
    json.dump(tags_by_length,f, indent = 3)

In [109]:
with open("tags_to_word_index.json","w") as f:
    json.dump(tags_to_word_index, f, indent=3)